# Module 6: Agent-as-Tool

Apply **Pattern 5**: wrap specialized agents as callable tools so an orchestrator can delegate like a manager to experts.

```
Decision Brief
       │
       ▼
┌──────────────────────────────────────────┐
│             ORCHESTRATOR                 │
│  tools = [researcher_agent,              │
│            analyzer_agent,               │
│            synthesizer_agent]            │
└───────────┬──────────────────────────────┘
            │ LLM decides: who, when, what parameters
            │
    ┌───────┼──────────┬──────────────┐
    ▼       ▼          ▼              ▼
researcher analyzer_A  analyzer_B  synthesizer
 @tool      @tool       @tool       @tool
 Agent(...)  Agent(...)  Agent(...)  Agent(...)
 silent     silent      silent      streams
```

**Key difference from Module 2 (Sequential Chain):** In Module 2 you wrote Python code to chain agents in order. Here the **LLM orchestrator decides** when to call each specialist, with what parameters, and in what order — no routing code needed.

**When to use this pattern:**
- Clear hierarchy: one coordinator, many experts
- Each specialist has a tailored prompt and its own tools
- Add/remove specialists without touching the orchestrator

**Key Strands API:** `@tool` decorator wrapping an `Agent`; the docstring is the routing logic.

**Prerequisites:** Modules 1–5.

## Tools and Components in This Module

| Component | Type | What it does |
|-----------|------|-------------|
| `get_company_data` | Tool | NovaCart financial/operational data |
| `get_market_benchmarks` | Tool | E-commerce industry benchmarks |
| `get_competitor_data` | Tool | Competitor premium tier details |
| `researcher_agent` | `@tool` wrapping Agent | Research market context — passes topic, returns structured findings |
| `analyzer_agent` | `@tool` wrapping Agent | Evaluate one option — passes name + description + research, returns assessment |
| `synthesizer_agent` | `@tool` wrapping Agent | Write the memo — passes brief + all analyses, returns final memo |
| `orchestrator` | `Agent(tools=[...])` | Coordinates the specialists — LLM decides routing |

> **The docstring IS the routing logic.** The orchestrator reads each tool's docstring to decide when to call it and what arguments to pass. Write docstrings for the model, not for developers.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# ── Model configuration ──────────────────────────────────────────────────
# Option 1 — Claude Sonnet 4 (default):
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
# Option 2 — Claude Haiku 4.5 (faster):
#   model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")
# Option 3 — Amazon Nova Pro (AWS credits):
#   model = BedrockModel(model_id="amazon.nova-pro-v1:0")
# Option 4 — Amazon Nova Lite (cheapest):
#   model = BedrockModel(model_id="amazon.nova-lite-v1:0")
print("✅ Setup complete!")

In [ ]:
import sys, os, time, json
sys.path.insert(0, os.path.join(os.getcwd(), "..", "01-strands-foundations"))

from strands import Agent, tool
from decision_brief_tools import get_company_data, get_market_benchmarks, get_competitor_data

---

## Part 1 — System Prompts

Each specialist agent has a **narrow, focused system prompt** — it does one job and nothing else.
This is what makes the pattern composable: swap a specialist by changing its system prompt.

In [ ]:
RESEARCHER_PROMPT = (
    "You are a market research specialist. Use your tools to gather relevant data about "
    "the company, industry benchmarks, and competitors. Return structured findings — data only."
)

ANALYZER_PROMPT = (
    "You are a business strategy analyst. Evaluate the ONE option you are given: "
    "strengths, weaknesses, implementation complexity (Low/Med/High), "
    "top 2 risks with specific mitigations, and a verdict (Proceed/Caution/Do not proceed). "
    "Be concise — 150 words max."
)

SYNTHESIZER_PROMPT = (
    "You are an executive communications specialist. "
    "Write a leadership decision memo from the brief and option analyses provided:\n"
    "## Recommendation (one sentence: which option and why)\n"
    "## Options at a Glance (table comparing A, B, C)\n"
    "## Top 3 Risks with mitigations\n"
    "## Success Metrics (KPIs with numeric targets)\n"
    "## Decision Required (owner, deadline, approvers)\n"
    "Under 400 words."
)

ORCHESTRATOR_PROMPT = (
    "You are a strategic decision analyst coordinating a team of specialists.\n"
    "Steps:\n"
    "1. Call researcher_agent to gather market data for the decision topic.\n"
    "2. Call analyzer_agent THREE times — once for each option (A, B, C) — passing the research context.\n"
    "3. Call synthesizer_agent with all three analyses to produce the final memo.\n"
    "Do not skip any step. Always analyze all three options."
)

---

## Part 2 — Wrap Specialists as `@tool`

The `@tool` decorator turns each agent into a callable tool. The orchestrator treats them exactly like any other tool — it reads the docstring to decide when and how to call each one.

**Three ways to use agents as tools in Strands:**

```python
# Option A — @tool decorator (most control, multi-parameter)
@tool
def researcher_agent(topic: str) -> str: ...

# Option B — pass Agent directly in tools[] (simplest, single input)
orchestrator = Agent(tools=[researcher_agent_instance, ...])

# Option C — .as_tool() (custom name/description, optional preserve_context)
orchestrator = Agent(tools=[researcher.as_tool(name="...", description="...")])
```

This module uses **Option A** — the `@tool` decorator gives us multi-parameter tools so the orchestrator can pass precise arguments (e.g., option name + description + research context) to the analyzer.

In [ ]:
@tool
def researcher_agent(topic: str) -> str:
    '''Research market context, company data, benchmarks, and competitive intelligence for a decision topic.

    Args:
        topic: The decision topic or brief to research
    '''
    worker = Agent(
        tools=[get_company_data, get_market_benchmarks, get_competitor_data],
        system_prompt=RESEARCHER_PROMPT,
        callback_handler=None,   # silent — passes its output to the orchestrator
    )
    return str(worker(topic))


@tool
def analyzer_agent(option_name: str, option_description: str, research_context: str) -> str:
    '''Analyze one specific decision option and return a structured assessment.

    Use this tool once per option. Call it three times total (Option A, B, C).

    Args:
        option_name: Short name of the option (e.g., "Option A — Exclusive Premium")
        option_description: Full description including price, approach, and timeline
        research_context: Market research findings from researcher_agent to inform the analysis
    '''
    worker = Agent(system_prompt=ANALYZER_PROMPT, callback_handler=None)
    return str(worker(
        f"Option: {option_name}\nDescription: {option_description}\nResearch: {research_context}"
    ))


@tool
def synthesizer_agent(decision_brief: str, all_analyses: str) -> str:
    '''Synthesize all option analyses into an executive leadership memo.
    Call this AFTER analyzer_agent has been called for all three options.

    Args:
        decision_brief: The original decision brief text
        all_analyses: Combined analyses of all three options from analyzer_agent
    '''
    worker = Agent(system_prompt=SYNTHESIZER_PROMPT)
    return str(worker(f"Brief:\n{decision_brief}\n\nOption analyses:\n{all_analyses}"))

---

## Part 3 — Build and Run the Orchestrator

The orchestrator is a standard `Agent` — but instead of business tools (like `get_company_data`), its tools are other agents. The LLM decides the routing, argument construction, and order of calls.

In [ ]:
orchestrator = Agent(
    tools=[researcher_agent, analyzer_agent, synthesizer_agent],
    system_prompt=ORCHESTRATOR_PROMPT,
)

DECISION_BRIEF = '''
DECISION BRIEF: NovaCart Premium Tier Launch

Options:
  Option A — Exclusive Premium: invite-only for top 10% of spenders, $19.99/mo
  Option B — Gradual Rollout: 5% A/B test pilot with kill-switch, $14.99/mo
  Option C — Full Launch: open to all users immediately, $12.99/mo + 30-day free trial

Success target: +15% CLV improvement within 6 months
Budget: $2M  |  Decision deadline: 2027-01-31
'''

print("Running orchestrator (LLM decides routing)...")
t0 = time.time()
result = orchestrator(DECISION_BRIEF)
elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s")

---

## Part 4 — Inspect What the Orchestrator Decided

Unlike Module 2 where the routing was Python code, here the routing lives in the orchestrator's `agent.messages`. Let's see exactly which tools it called, in what order, and with what parameters.

In [ ]:
print("=== ORCHESTRATOR TOOL CALLS ===")
call_count = 0
for msg in orchestrator.messages:
    for block in msg.get("content", []):
        if "toolUse" in block:
            tu = block["toolUse"]
            call_count += 1
            inp = json.dumps(tu.get("input", {}))
            print(f"  {call_count}. {tu['name']}({inp[:100]}...)")
print()
print(f"Total tool calls: {call_count}")
print("The orchestrator called researcher once, analyzer 3 times (A, B, C), synthesizer once.")
print("This routing was decided by the LLM — not by Python code.")

In [ ]:
# Token usage
summary = result.metrics.get_summary()
usage = summary.get("accumulated_usage", {})

print(f"{'Metric':<20} {'Value':>10}")
print("-" * 32)
print(f"{'Input tokens':<20} {usage.get('inputTokens', 0):>10,}")
print(f"{'Output tokens':<20} {usage.get('outputTokens', 0):>10,}")
print(f"{'Total tokens':<20} {usage.get('totalTokens', 0):>10,}")
print(f"{'LLM cycles':<20} {summary.get('total_cycles', 'n/a'):>10}")
print()

tool_usage = summary.get("tool_usage", {})
if tool_usage:
    print("Per-tool stats:")
    for name, data in tool_usage.items():
        s = data.get("execution_stats", {})
        print(f"  {name}: calls={s.get('call_count',0)} | avg_time={round(s.get('average_time',0),1)}s")

---

## Key Takeaways

| Concept | What you saw |
|---------|-------------|
| `@tool` wrapping `Agent` | The docstring is the routing logic — the orchestrator reads it |
| `callback_handler=None` | Silent sub-agents; only the orchestrator + final synthesizer stream |
| Multi-parameter `@tool` | The orchestrator constructs precise arguments per call |
| LLM routing | The orchestrator called researcher(1) → analyzer(3×) → synthesizer(1) — no Python routing |
| `result.metrics.tool_usage` | Shows call count and timing per specialist |

**vs Module 2 (Sequential Chain):**
- Module 2: Python code calls agents in fixed order
- Module 6: LLM orchestrator decides order, parameters, and which tool for each sub-task

---

## What's Next

**Module 7** is the capstone: the complete Decision-Memo System combining all 4 patterns — Parallel heads + Critic-Refiner + Agent-as-Tool specialists + Sequential synthesis.

---

> 💡 **Note — Memory & Observability (review):**
>
> **Memory:** Each specialist creates a fresh `Agent` instance per call — stateless by default. If you wanted the analyzer to remember Option A's analysis when evaluating Option B, use `.as_tool(preserve_context=True)`. Review: when is cross-call memory in a specialist beneficial vs harmful? (Module 6 Memory)
>
> **Observability:** With 5 sub-agent calls, you want per-specialist traces. Review: how would OTEL spans look for a nested agent call? Each `researcher_agent(...)` call would show as a tool span in the orchestrator trace, with its own inner loop trace. (Module 7)

---

## Run interactively

```bash
cd samples/06-agent-as-tool
pip install -r requirements.txt
python chat.py
```